# 06 — 增加时间依赖：从 GMM 到贝叶斯滤波
回到第 02 章的三个高斯簇。观测空间与三个中心不变，适当提高观测噪声以突出时间信息的作用，但现在隐变量不再独立采样，而是按照转移概率生成。第 02 章的逐样本独立先验现在由上一时刻的后验和转移矩阵共同决定。必须重新生成有时间顺序的数据。

主动定义初始概率 $p(x_0)$、转移矩阵 $A_{ij}=p(x_t=j|x_{t-1}=i)$ 和观测密度 $p(y_t|x_t)$。本章先假设参数已知，只做推断。真实类别仅用于检查模拟结果，不进入滤波算法。

递推预测：$p(x_t|y_{0:t-1})=\sum_i p(x_t|x_{t-1}=i)p(x_{t-1}=i|y_{0:t-1})$。

更新：$p(x_t|y_{0:t})\propto p(y_t|x_t)p(x_t|y_{0:t-1})$。归一化常数就是当前观测的预测密度。

In [ ]:
import numpy as np
from scipy.stats import multivariate_normal
import matplotlib.pyplot as plt
from utils import plot_likelihood
rng = np.random.default_rng(42)
mu = np.array([[-2.,0.],[2.,0.],[0.,3.]])
sigma = 1.1
K = 3
# 转移矩阵 A[i,j] = p(x_t=j | x_{t-1}=i)。
A = np.full((K,K),.025)
np.fill_diagonal(A,.95)
T = 300
# 仅用于仿真产生状态和观测；滤波不使用真实状态。
x = np.empty(T,dtype=int)
x[0] = rng.integers(K)
for t in range(1,T):
    x[t] = rng.choice(K,p=A[x[t-1]])
noise = rng.normal(size=(T,2))
y = np.zeros((T,2))
for t in range(T):
    y[t] = mu[x[t]] + sigma*noise[t]
# 与第 02 章相同：每个类别对应一个条件高斯分布。
models = []
for k in range(K):
    models.append(multivariate_normal(mu[k],sigma**2*np.eye(2)))
conditional = np.zeros((T,K))
for t in range(T):
    for k in range(K):
        conditional[t,k] = models[k].pdf(y[t])

def filter_sequence(conditional,A):
    # 初始状态先验 p(x_0) 为均匀分布。
    posterior = np.ones(K)/K
    filtered = []
    log_evidence = []
    for t in range(len(conditional)):
        # 预测：对上一时刻的状态求和，得到当前状态的预测先验。
        predicted = np.zeros(K)
        if t == 0:
            predicted = posterior.copy()
        else:
            for j in range(K):
                for i in range(K):
                    predicted[j] += A[i,j]*posterior[i]
        # 更新：沿用第 02 章的先验 × 条件密度。
        joint = np.zeros(K)
        for k in range(K):
            joint[k] = conditional[t,k]*predicted[k]
        # 边缘化：当前观测的预测密度 p(y_t | 历史观测)。
        evidence = sum(joint)
        # 贝叶斯归一化：得到当前状态后验。
        for k in range(K):
            posterior[k] = joint[k]/evidence
        filtered.append(posterior.copy())
        log_evidence.append(np.log(evidence))
    return np.array(filtered),np.array(log_evidence)

# 均匀转移意味着每个时刻独立，退化成静态 GMM。
static, _ = filter_sequence(conditional,np.ones((K,K))/K)
filtered, ll = filter_sequence(conditional,A)
for row in filtered:
    assert np.isclose(sum(row),1)
print('static accuracy:',(static.argmax(1)==x).mean())
print('filter accuracy:',(filtered.argmax(1)==x).mean())
fig,ax = plt.subplots(figsize=(10,3))
ax.plot(x[:100],label='truth')
ax.plot(static.argmax(1)[:100],'.',label='static')
ax.plot(filtered.argmax(1)[:100],'--',label='filter')
ax.legend()
plt.show()

## 滤波与参数学习的连接
滤波本身不估计转移参数，但每个时刻得到的归一化常数 $p(y_t|y_{0:t-1})$ 相乘就是序列似然。因此可以直接比较参数，甚至优化它。下面只估计自转移概率 $a$，其他参数固定。

与前面章节不同，观测不再独立，因此不能直接相加各自的静态 log p(y)。应使用概率链式法则，将每一步预测对数密度相加。这里只优化一个标量，用网格搜索代替梯度迭代，便于看清目标函数。

In [ ]:
candidates = np.linspace(.34,.995,100)
likelihood = []
for a in candidates:
    # 给定候选参数 a，构造转移矩阵。
    candidate = np.full((3,3),(1-a)/2)
    np.fill_diagonal(candidate,a)
    # 滤波负责计算各时间步的边缘对数密度。
    filtered_candidate, log_evidence = filter_sequence(conditional,candidate)
    likelihood.append(sum(log_evidence))
best_index = np.argmax(likelihood)
print('MLE of self-transition probability (grid):', candidates[best_index])
plot_likelihood(candidates, likelihood, .95, 'simulation truth')

改变观测噪声、转移概率，观察静态后验和滤波后验的差异。离散隐变量使预测中的边缘化变成有限求和；连续线性高斯模型则可用卡尔曼滤波实现同一个递推原理。